In [0]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [0]:
file_type = 'csv'
file_location = '/Volumes/workspace/it3388/raw_data/price_history/price_histories.csv'

price_histories_df = spark.read.format(file_type).option("header", "true").load(file_location)
display(price_histories_df.limit(10))

In [0]:
from pyspark.sql.functions import col

# Rename columns in the DataFrame and cast timestamp column to timestamp type
renamed_df = price_histories_df \
    .withColumnRenamed("timestamp (UTC)", "UTC_timestamp") \
    .withColumnRenamed("discount (%)", "percent_discount") \
    .withColumnRenamed("discounted price (USD)", "USD_discounted_price") \
    .withColumnRenamed("original price (USD)", "USD_original_price") \
    .withColumn("UTC_timestamp", col("UTC_timestamp").cast("timestamp")) \
    .withColumn("percent_discount", col("percent_discount").cast("float")) \
    .withColumn("USD_discounted_price", col("USD_discounted_price").cast("float")) \
    .withColumn("USD_original_price", col("USD_original_price").cast("float")) \


# Store the contents of the file as a table
renamed_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.it3388.bronze_price_histories")

In [0]:
# Read the table
bronze_price_histories_df = spark.table("workspace.it3388.bronze_price_histories")

In [0]:
# Print shape of the table
print(f"{bronze_price_histories_df.count()} rows, {len(bronze_price_histories_df.columns)} columns")

In [0]:
# Check number of missing values
from pyspark.sql.functions import count, when, col

bronze_price_histories_df.select([count(when(col(c).isNull(), c)).alias(c) for c in bronze_price_histories_df.columns]).show()

In [0]:
# Plot histogram of percent_discount
discounts = bronze_price_histories_df.select("percent_discount").dropna().toPandas()
plt.figure(figsize=(8, 5))
sns.histplot(discounts["percent_discount"], bins=30, kde=True)
plt.xlabel("Discount Percent")
plt.title("Histogram of Discount Percent")
plt.show()

In [0]:
# Look at rows where the discount is > 100
bronze_price_histories_df.select("*").filter(col("percent_discount") > 100).show(truncate=False)